# 16 - Dashboard Data Export
## ShopEase Europe | Sentiment Analysis Project - Phase 2 
**Objective:** Prepare a clean, business friendly dataset for the 
Power BI interactive dashboard, including a derived flag identifying 
reviews related to the delivery and driver complaint theme identified 
in notebook 13.

## Import Libaries

In [2]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Load the Dataset

In [3]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'production_preprocessed_reviews.csv')

df = pd.read_csv(PROCESSED_DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Dataset loaded: 21,055 rows x 11 columns
Columns: ['review_id', 'cleaned_review', 'preprocessed_text', 'sentiment', 'country', 'product_category', 'year', 'month', 'rating', 'review_length', 'word_count']


## Add Delivery Complaint Flag
Flagging reviews that mention delivery or driver related terms, 
identified in notebook 13 as the primary driver of the 2019 to 2022 
sentiment decline, enabling this theme to be filtered directly in 
the dashboard.

In [3]:
delivery_terms = ['delivery', 'driver', 'delivered', 'drivers', 'courier', 'package', 'box', 'shipping']

def flag_delivery_complaint(text):
    text = str(text).lower()
    return any(term in text for term in delivery_terms)

df['delivery_related'] = df['cleaned_review'].apply(flag_delivery_complaint)

print(f"Reviews flagged as delivery related: {df['delivery_related'].sum():,} ({df['delivery_related'].mean()*100:.1f}%)")
print(f"\nDelivery related flag by sentiment:")
print(df.groupby('sentiment')['delivery_related'].mean().round(3) * 100)

Reviews flagged as delivery related: 8,933 (42.4%)

Delivery related flag by sentiment:
sentiment
negative    46.7
neutral     46.0
positive    31.3
Name: delivery_related, dtype: float64


## Build Dashboard Ready Dataset
Selecting and renaming columns for clarity, dropping fields not 
needed for business reporting.

In [ ]:
dashboard_df = df[[
    'review_id', 'cleaned_review', 'sentiment', 'rating',
    'country', 'product_category', 'year', 'month',
    'review_length', 'delivery_related'
]].copy()
dashboard_df = dashboard_df.rename(columns={
    'cleaned_review': 'review_text',
    'product_category': 'category'
})

# Capitalise sentiment for cleaner display in Power BI
dashboard_df['sentiment'] = dashboard_df['sentiment'].str.capitalize()

print(f"Dashboard dataset shape: {dashboard_df.shape[0]:,} rows x {dashboard_df.shape[1]} columns")
print(f"\nColumns: {dashboard_df.columns.tolist()}")
display(dashboard_df.head())

Dashboard dataset shape: 21,055 rows x 10 columns

Columns: ['review_id', 'review_text', 'sentiment', 'rating', 'country', 'category', 'year', 'month', 'review_length', 'delivery_related']


,review_id,review_text,sentiment,rating,country,category,year,month,review_length,delivery_related
0,REV-50BCBCD9,"i registered on the website, tried to order a ...",Negative,1,US,Sports,2024,9,587,False
1,REV-6D2B2651,had multiple orders one turned up and driver h...,Negative,1,GB,Toys,2024,9,293,True
2,REV-F7E80372,i informed these reprobates that i would not b...,Negative,1,GB,Toys,2024,9,610,True
3,REV-ED2B173F,i have bought from amazon before and no proble...,Negative,1,AU,Sports,2024,9,449,False
4,REV-E48A7AB9,if i could give a lower rate i would! i cancel...,Negative,1,GB,Fashion,2024,9,535,False


## Export Dashboard Dataset

In [ ]:
DASHBOARD_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'dashboard_data.csv')

dashboard_df.to_csv(DASHBOARD_PATH, index=False)

print(f"Dashboard dataset exported successfully to:")
print(DASHBOARD_PATH)
print(f"\nFile size: {os.path.getsize(DASHBOARD_PATH) / (1024*1024):.2f} MB")

Dashboard dataset exported successfully to:
c:\Users\User\OneDrive\Desktop\shopease-sentiment-analysis\data\processed\dashboard_data.csv

File size: 10.32 MB


: 

## Complaint Theme Frequency Export
Calculating term frequency for key delivery and service complaint 
themes identified in notebook 13, calculated exclusively from 
negative reviews where the signal is strongest. This table will 
power the Top Complaint Themes chart on Power BI Page 2.

In [4]:
complaint_terms = {
    'delivery': ['delivery', 'deliver', 'delivered'],
    'driver': ['driver', 'drivers'],
    'damaged': ['damaged', 'damage', 'broken', 'defective'],
    'late / delayed': ['late', 'delay', 'delayed', 'overdue'],
    'missing package': ['missing', 'lost', 'never arrived', 'not received'],
    'refund': ['refund', 'refunded', 'money back'],
    'return': ['return', 'returned', 'sent back'],
    'customer service': ['customer service', 'support', 'helpline', 'chat']
}

negative_reviews = df[df['sentiment'] == 'negative']['cleaned_review'].str.lower()

theme_counts = {}
for theme, terms in complaint_terms.items():
    count = negative_reviews.apply(
        lambda text: any(term in text for term in terms)
    ).sum()
    theme_counts[theme] = count

complaint_df = pd.DataFrame(
    list(theme_counts.items()),
    columns=['complaint_theme', 'frequency']
).sort_values('frequency', ascending=False)

print("COMPLAINT THEME FREQUENCIES (Negative Reviews Only)")
print(complaint_df.to_string(index=False))

COMPLAINT_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'complaint_themes.csv')
complaint_df.to_csv(COMPLAINT_PATH, index=False)
print(f"\nExported to {COMPLAINT_PATH}")

COMPLAINT THEME FREQUENCIES (Negative Reviews Only)
 complaint_theme  frequency
customer service       5620
        delivery       5391
          refund       3558
  late / delayed       2548
          return       2249
 missing package       1596
          driver       1316
         damaged       1006

Exported to c:\Users\User\OneDrive\Desktop\shopease-sentiment-analysis\data\processed\complaint_themes.csv


## Customer Service Complaint Context

A secondary check revealed that 2,902 negative reviews mention customer 
service terms without any delivery context, representing 20% of all 
negative reviews. Review sampling identified three distinct sub-themes 
within this group: account suspension and billing disputes, marketplace 
seller trust issues, and service channel quality complaints including 
automated systems and response times.

**Insight:** Customer service is the highest frequency complaint theme 
overall, reflecting a persistent, multi-issue baseline problem rather 
than a single addressable root cause. This is distinct from the delivery 
theme, which showed the largest proportional growth during 2019 to 2022 
specifically. The two findings are complementary rather than 
contradictory: delivery operations drove the measurable decline in that 
period, while customer service represents a chronic, pre-existing 
dissatisfaction that predates and outlasts the delivery deterioration. 
Addressing both requires separate workstreams, operational delivery 
partner audits for the decline, and systematic customer service channel 
improvements for the baseline.

## Summary

Two datasets were exported to support the Power BI dashboard and 
downstream business reporting.

The primary export, dashboard_data.csv, contains 21,055 rows and 10 
columns, combining cleaned review text, sentiment labels, rating, 
country, category, temporal features, review length, and a derived 
delivery_related flag. This flag confirmed the notebook 13 finding at 
scale: 42.4% of all reviews mention delivery related terms, rising to 
46.7% among negative reviews compared to 31.3% among positive ones.

The secondary export, complaint_themes.csv, calculates term frequency 
for eight key complaint themes across negative reviews only. Customer 
service emerged as the highest frequency theme at 5,620 mentions, 
followed closely by delivery at 5,391. A secondary check revealed that 
2,902 of the customer service mentions occur without any delivery 
context, identifying three distinct sub-themes: account suspension and 
billing disputes, marketplace seller trust issues, and service channel 
quality complaints including automated systems and response times.

This distinction is analytically important. Customer service represents 
a persistent, multi-issue baseline problem across the entire dataset 
timeframe. Delivery terms, by contrast, showed the largest proportional 
growth specifically between 2019 and 2022, making them the more 
probable driver of the measurable sentiment decline during that period. 
The two findings are complementary rather than contradictory, and both 
warrant separate operational workstreams from ShopEase Europe's 
leadership.

### Business Impact
The complaint theme export gives ShopEase Europe's operations and 
customer experience teams a direct, quantified view of where customer 
pain is concentrated. Delivery and customer service together account 
for the overwhelming majority of complaint mentions in negative reviews. 
Treating these as a single problem would be a mistake. Delivery failures 
require operational intervention with last-mile partners and driver 
conduct standards. Customer service failures require systemic improvements 
to contact channels, response quality, and resolution capability. 
Addressing both simultaneously, with separate owners and separate 
metrics, is the recommendation that follows from this data.